# Ritratti IA — Andrea Badiali · FLUX.1 Kontext [dev] INT4 (Nunchaku) su Kaggle T4

**Uso:** solo landing page, con il consenso di Andrea. Nessuna ambientazione in luoghi, eventi o con persone reali.
Prompt e logica degli scatti: `foto_ai/prompt_kontext.md`.

**Prima di iniziare (Kaggle):**
- Session options → **Accelerator: GPU T4 x2** · **Internet: On**
- Add-ons → Secrets → `HF_TOKEN` **collegato a questo notebook**
- Input → il tuo dataset privato con le foto

**Come si usa:** esegui le celle in ordine. Dalla cella 7 in poi ogni cella mostra una griglia di varianti: nella cella *successiva* scrivi l'indice della migliore in `choose("...", INDICE)`.

> ⚠ VERIFICA = punto in cui nomi di repository/file o versioni potrebbero essere cambiati: controlla il README di Nunchaku (github.com/nunchaku-tech/nunchaku) e la model card su Hugging Face.

## 0 · Configurazione
L'unica cella da modificare (oltre alle scelte `choose`). `CROP_A` è il ritaglio 4:5 della foto sorgente A, in frazioni di larghezza/altezza: lo verifichi nell'anteprima della cella 4.

In [ ]:
# ==== PATHS ====
INPUT_DIR  = "/kaggle/input"
OUT_DIR    = "/kaggle/working/output"     # every generated candidate (+ .json metadata)
CHOSEN_DIR = "/kaggle/working/scelte"     # only the images you pick with choose()
RESUME_DIR = None                         # e.g. "/kaggle/input/badiali-scelte" to resume from a previous session

# ==== SOURCES ====
SRC_A_NAME = "SaveClip.App_584815494_18491809804076094_339424971745512787_n.jpg"  # microphone photo, 1440x1440
SRC_B_NAME = None   # high-res original of the black t-shirt / piano photo, when available
CROP_A = (0.375, 0.299, 0.753, 0.771)    # (left, top, right, bottom), 4:5 around Andrea

# ==== MODELS ====  ⚠ VERIFY on the model cards
BASE_MODEL    = "black-forest-labs/FLUX.1-Kontext-dev"
NUNCHAKU_REPO = "nunchaku-tech/nunchaku-flux.1-kontext-dev"           # formerly mit-han-lab/...
NUNCHAKU_FILE = "svdq-{precision}_r32-flux.1-kontext-dev.safetensors"

# ==== GENERATION ====
N_SEEDS   = 4
STEPS     = 28
GUIDANCE  = 2.5      # raise to 3.0-3.5 only if an instruction is ignored
BASE_SEED = 1000

USE_FACE_SCORE = False   # True = install insightface and show an identity similarity score per variant

IDENTITY = (
    "Keep the exact same person: same face shape, same facial features, same dark curly "
    "shoulder-length hair, same short beard and moustache, same round clear-frame glasses, "
    "same eye color and natural skin texture with visible pores. Do not beautify or retouch the face."
)

### Prompt
Una sola modifica per prompt. La formula `IDENTITY` viene aggiunta automaticamente in coda.

In [ ]:
PROMPTS = {
    # --- Shot 1: studio portrait (chain on source A) ---
    "1a": "Remove the black microphone from the man's hands; his hands are now resting relaxed and out of frame below the chest. Keep the pose, clothing, background and lighting unchanged.",
    "1b": "Replace the white t-shirt with a crisp white dress shirt with a classic collar, top button open, no tie, worn under the same navy blue tailored blazer. Keep everything else unchanged.",
    "1c": "Replace the background with a seamless warm grey studio backdrop with a soft gradient, slightly lighter behind the head. Keep the man, his clothing and his pose unchanged.",
    "1d": "Relight the portrait as a professional studio photo: large softbox key light from the left at 45 degrees creating soft Rembrandt lighting, gentle fill from the right, subtle rim light on the hair. Shot on an 85mm lens at f/2.8, shallow depth of field, warm muted color grade, low saturation, fine film grain. Keep the pose and clothing unchanged.",
    "1e": "Change only the expression: mouth closed in a calm, subtle closed-lip smile. Keep head angle, gaze direction, pose, clothing, background and lighting unchanged.",
    # --- Shot 2: hero ---
    "2":  "Convert to a black and white fine-art portrait: deep black background, single hard side light from the left leaving the right half of the face in soft shadow, high contrast chiaroscuro, rich blacks, subtle film grain, in the style of a classical music album cover. Keep the pose and clothing unchanged.",
    "2c": "Keep the colors, with a warm muted grade and amber key light: deep dark background, single side light from the left leaving the right half of the face in soft shadow, chiaroscuro, subtle film grain, in the style of a classical music album cover. Keep the pose and clothing unchanged.",
    # --- Shot 3: turtleneck ---
    "3":  "Replace the navy blazer and white shirt with a fine-knit charcoal grey turtleneck sweater. Keep the background, lighting, pose and framing unchanged.",
    # --- Shot 4: at work ---
    "4a": "Replace the background with a dim music room: an upright piano with dark wood finish behind the man, a warm table lamp, soft evening light. Keep the man, clothing and pose unchanged.",
    "4b": "The man is now holding a pencil and looking down at a handwritten music score on a desk in front of him, with sheets of manuscript paper covered in handwritten notes. Keep his face, glasses, hair and clothing unchanged.",
    # --- Shot 5: generic theater ---
    "5a": "Replace the background with the interior of an empty historic Italian opera house seen from the stalls: rows of red velvet seats, gilded balconies softly out of focus, warm amber stage light. No logos, no signage, no recognizable venue. Keep the man, his clothing and pose unchanged.",
    "5b": "The man is now seated in a red velvet theater seat, leaning slightly forward with his forearms on his knees, looking toward the camera. Keep his face, glasses, hair and clothing unchanged.",
}

## 1 · Controllo GPU, versioni e memoria
Si ferma se la GPU non è adatta (es. P100). Sulla T4 (Turing, capability 7.5) si lavora in **float16**: niente bf16.

In [ ]:
import sys, platform, shutil
import torch, psutil

print("Python :", platform.python_version())
print("Torch  :", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2"

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}  : {p.name} | capability {p.major}.{p.minor} | {p.total_memory/1e9:.1f} GB")

cap = torch.cuda.get_device_capability(0)
if cap < (7, 5):
    raise SystemExit(f"GPU capability {cap} not supported by Nunchaku (e.g. P100). Choose GPU T4 x2.")
print("Dtype  :", "float16 (Turing, no bf16)" if cap[0] < 8 else "bfloat16")
print("RAM    :", round(psutil.virtual_memory().total / 1e9, 1), "GB")
print("Disk   :", round(shutil.disk_usage("/").free / 1e9, 1), "GB free")
if torch.cuda.device_count() < 2:
    print("NOTE: single GPU -> the text encoder will run on CPU (slower prompt encoding, needs ~10 GB RAM).")

## 2 · Installazione (torch bloccato)
- Un file di *constraints* blocca torch/torchvision alla versione attuale: nessun pacchetto può cambiarli.
- La wheel di Nunchaku viene scelta **automaticamente** dalle GitHub Releases in base a versione di torch e Python. **Non** usare `pip install nunchaku`: su PyPI esiste un pacchetto diverso con lo stesso nome.
- Scelta manuale: apri github.com/nunchaku-tech/nunchaku/releases, cerca il file `nunchaku-X.Y.Z+torchA.B-cpXY-cpXY-linux_x86_64.whl` con **A.B** = versione di torch stampata sopra e **cpXY** = versione di Python (3.11 → cp311), copia il link in `NUNCHAKU_WHEEL_URL`.
- ⚠ VERIFICA nel README di Nunchaku la versione minima di diffusers richiesta e il supporto alle GPU Turing (serie 20 / T4).

In [ ]:
import subprocess, json, re, urllib.request
import importlib.metadata as md_
import torchvision

NUNCHAKU_WHEEL_URL = ""          # leave empty for automatic selection
ALLOW_PRERELEASE   = False
PACKAGES = ["diffusers>=0.34", "transformers", "accelerate", "sentencepiece", "protobuf",
            "peft", "einops", "huggingface_hub", "safetensors"]

TORCH_BEFORE = md_.version("torch")
CONSTRAINTS = "/tmp/constraints.txt"
with open(CONSTRAINTS, "w") as f:
    f.write(f"torch=={TORCH_BEFORE}\ntorchvision=={md_.version('torchvision')}\n")

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS, *args])

def find_nunchaku_wheel():
    tv = ".".join(TORCH_BEFORE.split("+")[0].split(".")[:2])
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    pattern = re.compile(rf"\+torch{re.escape(tv)}-{py}-{py}-linux_x86_64\.whl$")
    api = "https://api.github.com/repos/nunchaku-tech/nunchaku/releases?per_page=40"
    releases = json.load(urllib.request.urlopen(api))
    seen = []
    for r in releases:                      # newest first
        if r["draft"] or (r["prerelease"] and not ALLOW_PRERELEASE):
            continue
        for a in r["assets"]:
            if a["name"].endswith(".whl"):
                seen.append(a["name"])
                if pattern.search(a["name"]):
                    return a["browser_download_url"]
    print(f"No wheel for torch {tv} / {py}. Recent wheels found:")
    for n in sorted(set(seen))[-25:]:
        print("  ", n)
    raise SystemExit("Set NUNCHAKU_WHEEL_URL manually (see the markdown above).")

pip(*PACKAGES)
url = NUNCHAKU_WHEEL_URL or find_nunchaku_wheel()
print("Nunchaku wheel:", url)
pip("--no-deps", url)

# Sanity checks in a fresh process
torch_after = md_.version("torch")
if torch_after != TORCH_BEFORE:
    print(f"⚠ torch changed {TORCH_BEFORE} -> {torch_after}: Run -> Restart session, then reinstall.")
check = subprocess.run([sys.executable, "-c", "import nunchaku, diffusers; print(nunchaku.__version__, diffusers.__version__)"],
                       capture_output=True, text=True)
print("nunchaku / diffusers:", check.stdout.strip() or check.stderr[-1500:])
for p in ["diffusers", "transformers", "accelerate", "huggingface_hub"]:
    print(f"  {p}=={md_.version(p)}")   # copy these into PACKAGES to pin them once everything works

## 3 · Login a Hugging Face
Legge `HF_TOKEN` dai Secrets (il valore non viene mai stampato) e verifica l'accesso alla licenza di FLUX Kontext.

In [ ]:
from huggingface_hub import login, whoami, hf_hub_download

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    import os
    return os.environ[name]

login(token=get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])
hf_hub_download(BASE_MODEL, "model_index.json")
print("FLUX.1 Kontext license: OK")

## 4 · Foto sorgente
Corregge la rotazione EXIF e ritaglia la sorgente A in 4:5. Nell'anteprima il riquadro rosso è il ritaglio: se taglia male, modifica `CROP_A` nella cella 0 e riesegui.

Nota: nella foto A il volto è largo solo ~200 px. Un originale a risoluzione maggiore (dal fotografo) migliorerebbe molto la somiglianza.

In [ ]:
import glob, os
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

EXTS = (".jpg", ".jpeg", ".png", ".webp", ".jfif")
all_imgs = [p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True) if p.lower().endswith(EXTS)]
print(f"{len(all_imgs)} images in {INPUT_DIR}")

def load(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")

def find(name):
    hits = [p for p in all_imgs if os.path.basename(p) == name]
    return hits[0] if hits else None

path_a = find(SRC_A_NAME)
if path_a is None:
    path_a = max(all_imgs, key=lambda p: Image.open(p).size[0] * Image.open(p).size[1])
    print("SRC_A_NAME not found, using the largest image:", path_a)
full_a = load(path_a)
W, H = full_a.size
box = tuple(int(v) for v in (CROP_A[0]*W, CROP_A[1]*H, CROP_A[2]*W, CROP_A[3]*H))
SRC_A = full_a.crop(box)

preview = full_a.copy()
ImageDraw.Draw(preview).rectangle(box, outline=(255, 0, 0), width=max(3, W // 300))
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(preview); ax[0].set_title(f"A full {W}x{H}")
ax[1].imshow(SRC_A);   ax[1].set_title(f"A crop {SRC_A.size[0]}x{SRC_A.size[1]}")
for a in ax: a.axis("off")
plt.show()

SRC_B = None
if SRC_B_NAME:
    pb = find(SRC_B_NAME)
    SRC_B = load(pb) if pb else None
    print("SRC_B:", pb or "NOT FOUND")
REFERENCES = [im for im in (SRC_A, SRC_B) if im is not None]

## 5 · Caricamento del modello
- **GPU 0:** transformer FLUX Kontext INT4 (Nunchaku, ~7 GB) + VAE, in float16 con attention fp16 di Nunchaku.
- **GPU 1** (o CPU se c'è una sola GPU): text encoder CLIP + T5-XXL, usati solo per codificare i prompt. Separarli evita sia l'esaurimento della VRAM sia quello della RAM.
- Nessun CPU offload: con questa divisione tutto entra in memoria. I consumi vengono stampati dopo ogni componente.

In [ ]:
import gc
from diffusers import FluxKontextPipeline
from nunchaku import NunchakuFluxTransformer2dModel
from nunchaku.utils import get_precision

DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16

def mem(tag):
    gpus = " | ".join(f"GPU{i} {torch.cuda.memory_allocated(i)/1e9:.1f} GB" for i in range(torch.cuda.device_count()))
    print(f"[{tag}] {gpus} | RAM {psutil.virtual_memory().used/1e9:.1f} GB")

precision = get_precision()              # 'int4' on T4
transformer = NunchakuFluxTransformer2dModel.from_pretrained(
    f"{NUNCHAKU_REPO}/{NUNCHAKU_FILE.format(precision=precision)}", torch_dtype=DTYPE)
if DTYPE == torch.float16:
    transformer.set_attention_impl("nunchaku-fp16")   # ⚠ VERIFY: fp16 attention for Turing GPUs
mem("transformer")

pipe = FluxKontextPipeline.from_pretrained(
    BASE_MODEL, transformer=transformer,
    text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None,
    torch_dtype=DTYPE).to("cuda:0")
mem("pipeline (transformer + VAE)")

TE_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cpu"
TE_DTYPE = DTYPE if TE_DEVICE.startswith("cuda") else torch.bfloat16
text_pipe = FluxKontextPipeline.from_pretrained(
    BASE_MODEL, transformer=None, vae=None, torch_dtype=TE_DTYPE).to(TE_DEVICE)
mem(f"text encoders on {TE_DEVICE}")

_EMB = {}
@torch.no_grad()
def encode(prompt):
    if prompt not in _EMB:
        pe, ppe, _ = text_pipe.encode_prompt(prompt=prompt, prompt_2=None, device=TE_DEVICE, max_sequence_length=512)
        _EMB[prompt] = (pe.to("cuda:0", DTYPE), ppe.to("cuda:0", DTYPE))
    return _EMB[prompt]

## 6 · Funzioni
- `generate(...)` genera una variante e la salva come `NNN_step_seedXXXX.png` con un `.json` accanto (prompt, seed, parametri, tempo).
- `run_step(step, sorgente, prompt)` genera `N_SEEDS` varianti e mostra la griglia: input, riferimento originale, varianti numerate.
- `choose(step, indice, as_=...)` salva la variante scelta in `scelte/` e la rende disponibile come input degli step successivi.
- `get(nome)` recupera un'immagine scelta, anche da una sessione precedente (`RESUME_DIR`).

In [ ]:
import time
import numpy as np

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHOSEN_DIR, exist_ok=True)
CANDIDATES, CHOSEN, TIMES = {}, {}, []

def generate(image, prompt_key, seed, step_id, steps=STEPS, guidance=GUIDANCE):
    text = PROMPTS.get(prompt_key, prompt_key)
    full = f"{text} {IDENTITY}"
    pe, ppe = encode(full)
    t0 = time.time()
    out = pipe(image=image, prompt_embeds=pe, pooled_prompt_embeds=ppe,
               guidance_scale=guidance, num_inference_steps=steps,
               generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    dt = time.time() - t0
    TIMES.append(dt)
    if np.asarray(out).max() < 8:
        print("⚠ black image: NaN in float16 -> see 'Errori comuni' at the end")
    n = len(glob.glob(f"{OUT_DIR}/*.png")) + 1
    path = f"{OUT_DIR}/{n:03d}_{step_id}_seed{seed}.png"
    out.save(path)
    with open(path.replace(".png", ".json"), "w") as f:
        json.dump(dict(step=step_id, prompt=full, seed=seed, steps=steps, guidance=guidance,
                       size=out.size, seconds=round(dt, 1), model=NUNCHAKU_REPO), f, indent=2)
    return out, path

def get(name):
    if isinstance(name, Image.Image):
        return name
    if name in CHOSEN:
        return CHOSEN[name]
    for d in (CHOSEN_DIR, RESUME_DIR):
        if d and os.path.exists(f"{d}/{name}.png"):
            CHOSEN[name] = load(f"{d}/{name}.png")
            return CHOSEN[name]
    raise KeyError(f"'{name}' not chosen yet: run the previous step and choose()")

def show_grid(step_id, src, results):
    cols = 2 + len(results)
    fig, ax = plt.subplots(1, cols, figsize=(3.2 * cols, 4.2))
    ax[0].imshow(src);  ax[0].set_title("input")
    ax[1].imshow(SRC_A); ax[1].set_title("originale")
    for i, r in enumerate(results):
        t = f"[{i}] seed {r['seed']}"
        if r["score"] is not None:
            t += f"\nsim {r['score']:.2f}"
        ax[i + 2].imshow(r["image"]); ax[i + 2].set_title(t)
    for a in ax: a.axis("off")
    fig.suptitle(f"Step {step_id}"); plt.tight_layout(); plt.show()

def run_step(step_id, source, prompt_key=None, n_seeds=N_SEEDS, **kw):
    src = get(source)
    results = []
    for i in range(n_seeds):
        seed = BASE_SEED + i
        img, path = generate(src, prompt_key or step_id, seed, step_id, **kw)
        results.append(dict(seed=seed, image=img, path=path, score=face_score(img)))
        avg = sum(TIMES) / len(TIMES)
        print(f"{step_id} [{i+1}/{n_seeds}] {TIMES[-1]:.0f}s (avg {avg:.0f}s) -> {os.path.basename(path)}")
    CANDIDATES[step_id] = results
    show_grid(step_id, src, results)

def choose(step_id, idx, as_=None):
    img = CANDIDATES[step_id][idx]["image"] if step_id in CANDIDATES else get(step_id)
    for name in filter(None, (step_id, as_)):
        CHOSEN[name] = img
        img.save(f"{CHOSEN_DIR}/{name}.png")
    print(f"chosen {step_id}[{idx}]" + (f" as {as_}" if as_ else ""))
    return img

def face_score(img):
    return None   # replaced in the next cell if USE_FACE_SCORE

### 6b · (Opzionale) Punteggio di somiglianza del volto
Con `USE_FACE_SCORE = True` installa `insightface` (su CPU, senza toccare torch) e mostra sotto ogni variante la somiglianza con le foto originali (coseno, 0–1). Indicativamente: **> 0.6** stessa persona ben conservata, **0.4–0.6** da controllare a occhio, **< 0.4** scartare. È un aiuto, non sostituisce il confronto visivo.

In [ ]:
if USE_FACE_SCORE:
    pip("insightface", "onnxruntime")
    from insightface.app import FaceAnalysis
    FACE = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
    FACE.prepare(ctx_id=-1, det_size=(640, 640))

    def face_emb(img):
        faces = FACE.get(np.asarray(img)[:, :, ::-1])
        if not faces:
            return None
        f = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        return f.normed_embedding

    refs = [e for e in (face_emb(im) for im in REFERENCES) if e is not None]
    assert refs, "No face detected in the reference images"
    REF_EMB = np.mean(refs, axis=0); REF_EMB /= np.linalg.norm(REF_EMB)

    def face_score(img):
        e = face_emb(img)
        return None if e is None else float(np.dot(e, REF_EMB))

    print("Face score ready, reference self-check:", [round(face_score(im), 2) for im in REFERENCES])

---
# Generazione a catena
Ogni cella: **(1)** `choose(...)` dello step precedente con l'indice che preferisci dalla griglia sopra, **(2)** genera lo step nuovo.
Se il volto deriva, non correggere a valle: riesegui lo step precedente cambiando `BASE_SEED`, oppure scegli un'altra variante.

## 7 · Shot 1 — Ritratto da studio (colore)
### 7a · Rimozione del microfono
Step più delicato: controlla barba e linea della mascella.

In [ ]:
run_step("1a", SRC_A)

### 7b · Camicia bianca

In [ ]:
choose("1a", 0)        # <- index of the best variant above
run_step("1b", "1a")

### 7c · Sfondo da studio

In [ ]:
choose("1b", 0)
run_step("1c", "1b")

### 7d · Luce

In [ ]:
choose("1c", 0)
run_step("1d", "1c")

### 7e · (Opzionale) Espressione
Rischio medio: cambiare la bocca può alterare la somiglianza. Metti `DO_1E = True` solo se 7d non convince.

In [ ]:
DO_1E = False
choose("1d", 0, as_="S1")
if DO_1E:
    run_step("1e", "1d")

In [ ]:
if DO_1E:
    choose("1e", 0, as_="S1")   # overrides S1 with the expression variant

fig, ax = plt.subplots(1, 2, figsize=(9, 6))
ax[0].imshow(SRC_A); ax[0].set_title("originale")
ax[1].imshow(get("S1")); ax[1].set_title("S1")
for a in ax: a.axis("off")
plt.show()

## 8 · Shot 2 — Hero in stile ECM (B/N + variante a colori)
Lo scatto resta in **4:5**: allargarlo a 16:9 qui rimpicciolirebbe il volto e peggiorerebbe la somiglianza. Il formato orizzontale si ottiene nella pagina web, mettendo la foto a destra su un fondo nero a tutta larghezza.

In [ ]:
run_step("2", "S1")
run_step("2c", "S1")

In [ ]:
choose("2", 0, as_="S2")
choose("2c", 0, as_="S2c")

## 9 · Shot 3 — Dolcevita

In [ ]:
run_step("3", "S1")

In [ ]:
choose("3", 0, as_="S3")

## 10 · Shot 4 — Al lavoro: partitura e pianoforte
Usa la sorgente B se disponibile (ha già il pianoforte), altrimenti S1 con lo sfondo cambiato in 4a.
⚠ Rischio alto (cambiano posa, sguardo e mani): 8 varianti invece di 4. Le note sul pentagramma saranno spesso incoerenti: accettabile se fuori fuoco.

In [ ]:
if SRC_B is None:
    run_step("4a", "S1")
else:
    print("Using SRC_B as input for 4b: skip 4a")

In [ ]:
if SRC_B is None:
    choose("4a", 0, as_="S4_in")
    src4 = "S4_in"
else:
    src4 = SRC_B
run_step("4b", src4, n_seeds=8)

In [ ]:
choose("4b", 0, as_="S4")

## 11 · Shot 5 — Teatro generico

In [ ]:
DO_5B = False   # True = seated pose variant (medium-high identity risk)
run_step("5a", "S1")

In [ ]:
choose("5a", 0, as_="S5")
if DO_5B:
    run_step("5b", "S5")

In [ ]:
if DO_5B:
    choose("5b", 0, as_="S5")

## 12 · Riepilogo, checklist e download
Mostra i risultati finali accanto all'originale e crea `badiali_ritratti.zip` in `/kaggle/working` (scaricalo dalla sezione **Output** prima di chiudere la sessione, oppure *Save Version → Quick Save*).
Per riprendere in un'altra sessione: carica la cartella `scelte/` come dataset privato e imposta `RESUME_DIR`.

In [ ]:
finals = [n for n in ("S1", "S2", "S2c", "S3", "S4", "S5") if n in CHOSEN]
fig, ax = plt.subplots(1, len(finals) + 1, figsize=(3.5 * (len(finals) + 1), 4.6))
ax[0].imshow(SRC_A); ax[0].set_title("originale")
for a, n in zip(ax[1:], finals):
    s = face_score(CHOSEN[n])
    a.imshow(CHOSEN[n]); a.set_title(n + (f"  sim {s:.2f}" if s is not None else ""))
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

print("""Checklist before publishing:
 [ ] glasses: same shape and frame as the original
 [ ] hair: same length, volume and curls
 [ ] beard and moustache: same density, not 'cleaned up'
 [ ] natural skin texture, no plastic look
 [ ] hands: five fingers, correct proportions
 [ ] no logos, text or recognizable venues
 [ ] color grade consistent with the other photos on the site
 [ ] FINAL APPROVAL FROM ANDREA on every published image""")

shutil.make_archive("/kaggle/working/badiali_ritratti", "zip", "/kaggle/working", "scelte")
print(f"Zip: /kaggle/working/badiali_ritratti.zip | all candidates in {OUT_DIR} | avg {np.mean(TIMES):.0f}s/image")

---
## Errori comuni

| Errore | Causa | Soluzione |
|---|---|---|
| `401 Unauthorized` | token errato o revocato | rigenera il token su HF e aggiorna il secret |
| `403` / `GatedRepoError` | licenza FLUX Kontext non accettata, o token di un altro account | accetta la licenza con lo **stesso** account del token |
| `SecretNotFound` | secret non collegato | Add-ons → Secrets → spunta `HF_TOKEN` |
| `ConnectionError` | Internet spento | Session options → Internet On (richiede verifica telefono) |
| `No wheel for torch X / cpXY` | nessuna wheel di Nunchaku per questa combinazione | scegli a mano in `NUNCHAKU_WHEEL_URL`; se non esiste, *Pin to original environment* su una versione con torch supportato |
| `ImportError` / `undefined symbol` importando nunchaku | wheel per un'altra versione di torch | controlla che `+torchA.B` nel nome della wheel coincida con torch stampato nella cella 1 |
| il pacchetto `nunchaku` non ha `NunchakuFluxTransformer2dModel` | installato da PyPI | `pip uninstall -y nunchaku` e reinstalla dalla cella 2 |
| `torch changed` | un pacchetto ha forzato un'altra versione | Run → Restart session e rieseguire; i constraints dovrebbero impedirlo |
| `CUDA out of memory` | risoluzione troppo alta o altri modelli in VRAM | Restart session; verifica che i text encoder siano su GPU1/CPU (cella 5) |
| sessione crashata per RAM | text encoder su CPU con poca RAM (Colab) | usa Kaggle T4 x2 |
| immagini nere | NaN in float16 | assicurati che `set_attention_impl("nunchaku-fp16")` sia attivo; ⚠ VERIFICA nel README le note per GPU Turing |
| `functional_tensor` (solo upscaling) | basicsr vecchio con torchvision recente | riguarda il notebook di upscaling, non questo |
| quota GPU esaurita | ~30 h/settimana su Kaggle (⚠ VERIFICA) | ferma la sessione quando non generi; lavora con `RESUME_DIR` |